# 03 — Team Risk Scoring

Each team is scored across four dimensions and ranked by overall risk.

**Learning notes:**
- A weighted sum is a simple but effective proxy for risk. The weights encode our judgement about what matters most.
- `total_risk_score` is a **relative ranking tool** — it tells you which teams need attention most, not an absolute measure of danger.
- The formula lives in `src/helpers.py` so changing the weights affects all notebooks consistently.

**Risk score formula:**
```
total_risk_score =
  (incident_count       × 1) +   ← raw volume
  (severity_score       × 2) +   ← weighted by P1-P4
  (open_ri_count        × 3) +   ← unresolved work
  (control_gap_exposure × 4)     ← RIs with no control mapped (highest weight)
```

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.helpers import load_data, set_plot_style, save_processed, compute_team_risk

set_plot_style()

data = load_data("../data/raw")
incidents = data["incidents"]
ris       = data["ris"]
controls  = data["controls"]
mappings  = data["mappings"]

print("Data loaded.")

## 1. Compute Team Risk Scores

In [ ]:
risk_df = compute_team_risk(incidents, ris, mappings)
print(f"{len(risk_df)} teams scored.")
display(risk_df)

## 2. Total Risk Score by Team

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
risk_df["total_risk_score"].sort_values().plot(kind="barh", ax=ax, color="coral")
ax.set_title("Team Risk Score  (higher = more risk)")
ax.set_xlabel("Total Risk Score")
plt.tight_layout()
plt.savefig("../data/processed/chart_team_risk_scores.png", dpi=120)
plt.show()

## 3. Risk Breakdown Heatmap

This heatmap shows WHY each team is risky — the colour is normalised (0–1 per column) so you can compare dimensions on the same scale, and the annotation shows the raw value.

In [ ]:
heatmap_cols = ["incident_count", "severity_score", "open_ri_count", "control_gap_exposure"]
heatmap_norm = risk_df[heatmap_cols].copy()

for col in heatmap_cols:
    col_max = heatmap_norm[col].max()
    if col_max > 0:
        heatmap_norm[col] = heatmap_norm[col] / col_max

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(
    heatmap_norm,
    annot=risk_df[heatmap_cols],
    fmt="d",
    cmap="YlOrRd",
    ax=ax,
    cbar_kws={"label": "Normalised Score (0–1)"},
)
ax.set_title("Team Risk Breakdown  (colour = normalised, annotation = raw value)")
ax.set_xlabel("Risk Dimension")
plt.tight_layout()
plt.savefig("../data/processed/chart_team_risk_heatmap.png", dpi=120)
plt.show()

## 4. Top 5 and Bottom 5 Teams

In [ ]:
print("Top 5 highest risk teams:")
display(risk_df.head(5))

print("\nTop 5 lowest risk teams:")
display(risk_df.tail(5))

## 5. Save Team Risk Scores

In [ ]:
path = save_processed(risk_df, "team_risk_scores.csv", "../data/processed")
print(f"Saved: {path}")

## Key Findings

*(Fill in after running)*

- **Highest risk team:** 
- **Why (which dimension drives it):** 
- **Lowest risk team:** 
- **Teams with high control gap exposure:** 

---
**Next:** Run `04_themes.ipynb` to identify recurring keywords across incident and RI text.